In [1]:
# Install dependencies
!pip install azure-ai-projects azure-core azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 6.3 MB/s eta 0:00:00


In [5]:
from azure.ai.projects import AIProjectClient
from azure.core.credentials import AzureKeyCredential
from openai import OpenAI
from google.colab import userdata

# Credentials
api_key = userdata.get("New_Open_AI")
endpoint = "https://azure-ai-email-123.services.ai.azure.com/api/projects/Default"

# Azure AI Project Client
client = AIProjectClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

# OpenAI Client
openai_client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)

# -------------------------------
# 1. INPUT: Medical Report
# -------------------------------
medical_report = input("Paste Medical Report: ")

# -------------------------------
# 2. CALL DEPLOYED AGENT
# -------------------------------
response = openai_client.responses.create(
    input=[{"role": "user", "content": medical_report}],
    extra_body={
        "agent_reference": {
            "name": "medical-agent",
            "version": "1",
            "type": "agent_reference"
        }
    },
)

print("--- INTERPRETED REPORT ---")
print(response.output_text)

Paste Medical Report: Patient Name: Neha Singh Age: 28 Gender: Female  Report: All blood parameters within normal range. Hemoglobin: 13.5 g/dL Blood Sugar: 90 mg/dL Cholesterol: Normal  Doctor Notes: Healthy individual. Maintain current lifestyle.
--- INTERPRETED REPORT ---
{
  "Patient_Name": "Neha Singh",
  "Detected_Conditions": [
    "Unknown"
  ],
  "Key_Findings": [
    "All blood parameters are within normal range",
    "Hemoglobin is 13.5 g/dL",
    "Blood sugar is 90 mg/dL",
    "Cholesterol is normal",
    "Doctor notes indicate a healthy individual"
  ],
  "Risk_Level": "Low",
  "Suggested_Actions": [
    "Maintain current lifestyle",
    "Continue healthy habits",
    "Follow up with a doctor if new symptoms appear"
  ]
}


In [6]:
# -------------------------------
# 3. STRUCTURED INTERPRETATION
# -------------------------------
client = OpenAI(
    api_key=api_key,
    base_url=f"{endpoint}/openai/v1",
)

medical_data = """
Patient: John Doe
Age: 45
Report: Blood test shows high cholesterol and slightly elevated sugar levels.
Notes: Risk of diabetes and cardiovascular issues.
"""

try:
    response = client.responses.create(
        model="gpt-5.4-mini",
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Interpret this medical report into JSON with keys: Patient_Name, Condition, Risk_Level, Suggested_Action. Return ONLY valid JSON."
                    },
                    {
                        "type": "input_text",
                        "text": f"MEDICAL REPORT:\n{medical_data}"
                    }
                ]
            }
        ]
    )

    print("--- STRUCTURED OUTPUT ---")
    print(response.output_text)

except Exception as e:
    print(f"Interpretation Error: {e}")

--- STRUCTURED OUTPUT ---
{"Patient_Name":"John Doe","Condition":"High cholesterol and slightly elevated sugar levels","Risk_Level":"Moderate","Suggested_Action":"Consult a healthcare provider for evaluation, lifestyle changes, and monitoring for diabetes and cardiovascular risk"}


In [7]:
# 4. PARSE JSON SAFELY
# -------------------------------
import json
import re

try:
    raw_text = response.output_text

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        raise ValueError("No valid JSON found")

    json_string = match.group(0)

    extracted_data = json.loads(json_string)

    print("--- PARSED DATA ---")
    print(json.dumps(extracted_data, indent=2))

except Exception as e:
    print(f"Parsing Error: {e}")


--- PARSED DATA ---
{
  "Patient_Name": "John Doe",
  "Condition": "High cholesterol and slightly elevated sugar levels",
  "Risk_Level": "Moderate",
  "Suggested_Action": "Consult a healthcare provider for evaluation, lifestyle changes, and monitoring for diabetes and cardiovascular risk"
}


In [8]:

# -------------------------------
# 5. STORE IN AZURE BLOB
# -------------------------------
from azure.storage.blob import BlobServiceClient
import datetime

blob_conn_str = userdata.get("bcs")

blob_service_client = BlobServiceClient.from_connection_string(blob_conn_str)

file_name = f"medical_report_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

blob_client = blob_service_client.get_blob_client(
    container="medical-output",   # changed container name
    blob=file_name
)

blob_client.upload_blob(response.output_text, overwrite=True)

print("Saved to storage:", file_name)

Saved to storage: medical_report_20260421_070222.txt
